In [1]:
import sys
import functools
import threading

def debug_flow(func):

    thread_local = threading.local()

    def tracer(frame, event, arg):

        if not hasattr(thread_local, "last_locals"):
            thread_local.last_locals = {}

        func_name = frame.f_code.co_name

        if event == "call":

            print(f"\nCALL → {func_name}")
            if frame.f_locals:
                print("ARGS:", frame.f_locals)

        elif event == "line":

            current = frame.f_locals
            previous = thread_local.last_locals

            for key, value in current.items():

                if key not in previous:
                    print(f"CREATE {func_name}.{key} = {value}")

                elif previous[key] != value:
                    print(
                        f"UPDATE {func_name}.{key}: "
                        f"{previous[key]} → {value}"
                    )

            thread_local.last_locals = current.copy()

        elif event == "return":

            print(f"RETURN ← {func_name} → {arg}")

        return tracer

    @functools.wraps(func)
    def wrapper(*args, **kwargs):

        sys.settrace(tracer)

        try:
            return func(*args, **kwargs)
        finally:
            sys.settrace(None)

    return wrapper

In [2]:
@debug_flow
def calculate(x):

    a = x + 10
    b = helper(a)

    result = b * 2

    return result


def helper(v):

    temp = v + 5
    return temp

In [3]:
calculate(5)


CALL → calculate
ARGS: {'x': 5}
CREATE calculate.x = 5
CREATE calculate.a = 15

CALL → helper
ARGS: {'v': 15}
CREATE helper.v = 15
CREATE helper.temp = 20
RETURN ← helper → 20
CREATE calculate.x = 5
CREATE calculate.a = 15
CREATE calculate.b = 20
CREATE calculate.result = 40
RETURN ← calculate → 40


40